# Web Scraping & Product Price Analysis – Air Conditioners (Amazon India)
**Stage 4 – Data Validation & Cleaning** (simplified, beginner version)

Pipeline position: **4 of 8**

## Objective
Find and fix problems in the data: duplicates, wrong data types, impossible values, wrong MRP, and outliers.

**Cleaning rules we follow**
- Missing prices/ratings stay **blank** (we never fill them with the average, because that would distort results).
- Impossible values (e.g. selling price > MRP) are removed.
- Outliers are only **flagged**, not deleted.

## Step 1 – Setup

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Folder that holds all the CSV files of the project.
# (In Google Colab: upload the CSV files and change this to "/content/")
DATA = "../0. DataSet/"

print("Setup complete.")

Setup complete.


In [16]:
df = pd.read_csv("Air_Conditioners_Extracted.csv")
start = df.copy()          # copy kept for the before / after comparison
print("Loaded:", df.shape)

Loaded: (32, 16)


## Step 2 – Missing values

In [17]:
missing = df.isna().sum()
missing[missing > 0]

,0
discount_price,1
actual_price,1


## Step 3 – Duplicates

In [18]:
print("Exact duplicate rows    :", df.duplicated().sum())
print("Duplicate ASIN          :", df["asin"].duplicated().sum())

before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="asin")
df = df.reset_index(drop=True)
print("Rows removed:", before - len(df), "| rows remaining:", len(df))

Exact duplicate rows    : 0
Duplicate ASIN          : 0
Rows removed: 0 | rows remaining: 32


## Step 4 – Correct data types
`pd.to_numeric(..., errors="coerce")` turns text into numbers; anything that is not a number becomes blank (NaN).

In [19]:
number_columns = ["ratings", "no_of_ratings", "discount_price", "actual_price", "tonnage", "energy_star"]

for col in number_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[number_columns].dtypes

,0
ratings,float64
no_of_ratings,float64
discount_price,float64
actual_price,float64
tonnage,float64
energy_star,float64


## Step 5 – Tidy the text columns
`strip()` removes extra spaces at the start/end, `lower()` and `title()` make the case consistent.

In [20]:
text_columns = ["name", "main_category", "sub_category", "brand", "ac_type"]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

df["main_category"] = df["main_category"].str.lower()
df["sub_category"] = df["sub_category"].str.title()

print("Brands:", sorted(df["brand"].dropna().unique()))

Brands: ['Acerpure', 'Blue Star', 'Carrier', 'Cruise', 'Daikin', 'Godrej', 'Helium', 'Hitachi', 'IFB', 'LG', 'Lloyd', 'Oakter', 'Panasonic', 'Samsung', 'Sharp', 'Voltas']


## Step 6 – Impossible values (range checks)
Each line finds rows that break a rule, counts them, and sets the wrong value to blank.

In [21]:
bad_rating = (df["ratings"] < 1) | (df["ratings"] > 5)
bad_reviews = df["no_of_ratings"] < 0
bad_price = (df["discount_price"] < 5000) | (df["discount_price"] > 300000)
bad_tonnage = (df["tonnage"] < 0.4) | (df["tonnage"] > 10)
bad_star = (df["energy_star"] < 1) | (df["energy_star"] > 5)

print("ratings outside 1-5            :", bad_rating.sum())
print("negative number of ratings     :", bad_reviews.sum())
print("price outside Rs 5,000-3,00,000:", bad_price.sum())
print("tonnage outside 0.4-10         :", bad_tonnage.sum())
print("energy star outside 1-5        :", bad_star.sum())

df.loc[bad_rating, "ratings"] = np.nan
df.loc[bad_reviews, "no_of_ratings"] = np.nan
df.loc[bad_price, "discount_price"] = np.nan
df.loc[bad_tonnage, "tonnage"] = np.nan
df.loc[bad_star, "energy_star"] = np.nan

ratings outside 1-5            : 0
negative number of ratings     : 0
price outside Rs 5,000-3,00,000: 0
tonnage outside 0.4-10         : 0
energy star outside 1-5        : 0


## Step 7 – Selling price must not be higher than MRP
The MRP (maximum retail price) is the highest price allowed. If the selling price is *higher*, one of them was scraped wrongly. We blank the MRP in those rows.

In [22]:
bad_mrp = df["actual_price"] < df["discount_price"]
print("Rows with selling price > MRP:", bad_mrp.sum())
df.loc[bad_mrp, ["name", "discount_price", "actual_price"]].head()

Rows with selling price > MRP: 6


,name,discount_price,actual_price
2,"Blue Star 1.5 Ton 3 Star, New Star Rated, Inve...",34990.0,17495.0
3,"Acerpure Chill Neo 1.5 Ton 3 star, 5280W Max C...",27490.0,13745.0
5,"Sharp 0.95 Ton 3 star, New Star Rated,Split AC...",24940.0,12470.0
18,"Blue Star 1.5 Ton 3 Star,New Star Rated, Wi-Fi...",37490.0,18745.0
23,"Daikin 1.5 Ton 3 Star, New Star rated, Inverte...",37490.0,18745.0


In [23]:
df.loc[bad_mrp, "actual_price"] = np.nan
print("Invalid MRP values removed:", bad_mrp.sum())

Invalid MRP values removed: 6


## Step 8 – Discount amount and discount percent
Calculated only where both prices are valid.

In [24]:
df["discount_amount"] = df["actual_price"] - df["discount_price"]
df["discount_percent"] = df["discount_amount"] / df["actual_price"] * 100

# a discount must be between 0 and 100 %
bad_percent = (df["discount_percent"] < 0) | (df["discount_percent"] > 100)
df.loc[bad_percent, "discount_amount"] = np.nan
df.loc[bad_percent, "discount_percent"] = np.nan

print("Listings with a verified discount:", df["discount_percent"].notna().sum(), "of", len(df))
df[["discount_price", "actual_price", "discount_amount", "discount_percent"]].describe().round(2)

Listings with a verified discount: 25 of 32


,discount_price,actual_price,discount_amount,discount_percent
count,31.00,25.00,25.00,25.00
mean,34483.13,58959.56,24688.64,40.25
std,7635.09,16505.25,10956.28,9.70
min,16999.00,27499.00,4240.00,11.86
25%,30140.00,47900.00,16910.00,38.18
50%,34990.00,60990.00,24000.00,42.30
75%,37490.00,68990.00,32991.00,46.59
max,49804.00,90790.00,44500.00,52.54


## Step 9 – Helper flag and remaining blanks

In [25]:
df["price_available"] = df["discount_price"].notna()

remaining = df.isna().sum()
print("Blank values left per column:")
print(remaining[remaining > 0])

Blank values left per column:
discount_price      1
actual_price        7
discount_amount     7
discount_percent    7
dtype: int64


## Step 10 – Flag price outliers (IQR method)
- Q1 = 25% of prices are below this, Q3 = 75% are below this, IQR = Q3 − Q1.
- A price is an outlier if it is below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR`.
We only **flag** them.

In [26]:
prices = df["discount_price"].dropna()
q1 = prices.quantile(0.25)
q3 = prices.quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

df["price_outlier"] = (df["discount_price"] < low) | (df["discount_price"] > high)

print("Q1 =", round(q1), " Q3 =", round(q3), " IQR =", round(iqr))
print("Normal price range:", round(low), "to", round(high))
print("Outliers flagged:", df["price_outlier"].sum())
df.loc[df["price_outlier"], ["name", "discount_price"]]

Q1 = 30140  Q3 = 37490  IQR = 7350
Normal price range: 19115 to 48515
Outliers flagged: 3


,name,discount_price
12,Oakter 0.5 Ton Inverter Studio AC 5000 (2026 M...,16999.0
17,Daikin 1.5 Ton 5 Star Inverter Split AC (Coppe...,48990.0
30,"Blue Star 2 Ton 3 Star, New Star Rated, Invert...",49804.0


## Step 11 – Before / after comparison

In [27]:
before_after = pd.DataFrame({
    "Before": [len(start), (start["actual_price"] < start["discount_price"]).sum(),
               start["discount_price"].isna().sum(), start["actual_price"].isna().sum(),
               start["tonnage"].isna().sum(), start["energy_star"].isna().sum()],
    "After":  [len(df), (df["actual_price"] < df["discount_price"]).sum(),
               df["discount_price"].isna().sum(), df["actual_price"].isna().sum(),
               df["tonnage"].isna().sum(), df["energy_star"].isna().sum()],
}, index=["Rows", "Selling price > MRP", "Missing selling price", "Missing MRP", "Missing tonnage", "Missing energy_star"])
before_after

,Before,After
Rows,32,32
Selling price > MRP,6,0
Missing selling price,1,1
Missing MRP,1,7
Missing tonnage,0,0
Missing energy_star,0,0


## Step 12 – Save the cleaned data

In [28]:
final_columns = ["name", "main_category", "sub_category", "image", "link", "ratings", "no_of_ratings",
                 "discount_price", "actual_price", "discount_amount", "discount_percent",
                 "tonnage", "energy_star", "brand", "asin", "ac_type", "is_inverter", "has_wifi",
                 "price_available", "price_outlier"]

clean_df = df[final_columns]
clean_df.to_csv("Air_Conditioners_Cleaned.csv", index=False)
print("Saved:", clean_df.shape)
clean_df.head()

Saved: (32, 20)


,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price,discount_amount,discount_percent,tonnage,energy_star,brand,asin,ac_type,is_inverter,has_wifi,price_available,price_outlier
0,Helium Smart Choice Air Smart Inverter Split A...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51diNudbdO...,https://www.amazon.in/Helium-Inverter-Capacity...,3.3,212.0,22990.0,40100.0,17110.0,42.668329,0.80,3.0,Helium,B0GZVYF7N5,Split,True,False,True,False
1,IFB 0.95 Ton 3 Star 2026 AI Powered Inverter S...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51M-MqPqBr...,https://www.amazon.in/IFB-Conditioner-Compress...,4.4,200.0,30990.0,50990.0,20000.0,39.223377,0.95,3.0,IFB,B0GSJV6XTF,Split,True,False,True,False
2,"Blue Star 1.5 Ton 3 Star, New Star Rated, Inve...",appliances,Air Conditioners,https://m.media-amazon.com/images/I/61-yR4Rid4...,https://www.amazon.in/Blue-Star-Inverter-Comfo...,4.9,12.0,34990.0,NaN,NaN,NaN,1.50,3.0,Blue Star,B0H87QTXQC,Split,True,False,True,False
3,"Acerpure Chill Neo 1.5 Ton 3 star, 5280W Max C...",appliances,Air Conditioners,https://m.media-amazon.com/images/I/51j8RwfTT-...,https://www.amazon.in/Acerpure-Neo1-5-AC5IPG61...,4.4,76.0,27490.0,NaN,NaN,NaN,1.50,3.0,Acerpure,B0GL2LWCVL,Split,True,False,True,False
4,"Hitachi Smart Choice 1.5 Ton 3 Star, New Star ...",appliances,Air Conditioners,https://m.media-amazon.com/images/I/51q3Ja02z0...,https://www.amazon.in/Hitachi-Inverter-Copper-...,4.1,478.0,36500.0,63990.0,27490.0,42.959837,1.50,3.0,Hitachi,B0GP71KXP5,Split,True,False,True,False


**Output of Stage 4:** `Air_Conditioners_Cleaned.csv`.  **Next → Stage 5: Data Aggregation & Representation.**